# 01 — Public wedge: column lineage

The **canonical wedge walkthrough**. You will see raw inputs, how ClearMetric standardizes them, what compile produces, and one impact example.

**SQL chain:** `raw_orders` → `orders_base` → `customer_totals` → `customers_report`

Next: [02 compile formats](02_compile_formats.ipynb) · [03 impact analysis](03_impact_analysis.ipynb)

In [ ]:
import importlib.util
import os
import sys
import urllib.error
import urllib.request
from pathlib import Path
from types import ModuleType

_COLD_START_GITHUB_RAW_BASE = 'https://raw.githubusercontent.com/ClearMetric-Labs/ClearMetric-Core/main'

_CACHED_NOTEBOOKS_DIR = Path('/Users/jonkim/.cache/clearmetric/github-main/examples/notebooks')

def _github_raw_base() -> str:
    paths = sys.modules.get("_paths")
    default = (
        paths.GITHUB_RAW_BASE if paths is not None else _COLD_START_GITHUB_RAW_BASE
    )
    return os.environ.get("CM_CLEARMETRIC_GITHUB_RAW_BASE", default)
def _fetch_repo_file(repo_relative: str, dest: Path) -> None:
    if dest.is_file():
        return
    paths = sys.modules.get("_paths")
    if paths is not None:
        paths._fetch_github_file(repo_relative, dest)
        return
    url = f"{_github_raw_base()}/{repo_relative}"
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with urllib.request.urlopen(url, timeout=60) as response:
            dest.write_bytes(response.read())
    except urllib.error.URLError as exc:
        raise FileNotFoundError(
            f"Failed to download {url}. Check network access and branch path."
        ) from exc
def _find_local_helpers(start: Path | None = None) -> Path | None:
    start = start or Path.cwd()
    for root in (start, *start.parents):
        nested = root / "examples" / "notebooks"
        if (nested / "_paths.py").is_file():
            return nested
        if (root / "_paths.py").is_file() and (root / "_notebook_setup.py").is_file():
            return root
    return None
def _resolve_setup_file(start: Path | None = None) -> Path:
    local = _find_local_helpers(start)
    if local is not None:
        return local / "_notebook_setup.py"
    paths = sys.modules.get("_paths")
    cache_dir = (
        paths.CACHED_NOTEBOOKS_DIR if paths is not None else _CACHED_NOTEBOOKS_DIR
    )
    setup_file = cache_dir / "_notebook_setup.py"
    if not setup_file.is_file():
        _fetch_repo_file("examples/notebooks/_notebook_setup.py", setup_file)
    return setup_file
def _exec_setup_module(setup_file: Path) -> ModuleType:
    spec = importlib.util.spec_from_file_location("_notebook_setup", setup_file)
    if spec is None or spec.loader is None:
        raise ImportError(f"Cannot load {setup_file}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module
_exec_setup_module(_resolve_setup_file()).setup_notebook()


In [ ]:
from _paths import lineage_demo_project, show_raw_sources

PROJECT_DIR = lineage_demo_project()
print(f"project: {PROJECT_DIR}")


## Raw inputs

Files on disk **before** adapters run. Paths match `discover()`.

In [ ]:
show_raw_sources(PROJECT_DIR)

## Standardize

`discover` → `ingest_all` → `build_graph` — same APIs as `cm scan` / `cm compile`.

The warehouse export is a realistic Shopify raw zone (22 tables). The SQL folder adds three logical models not in that export. `warehouse_bind_unresolved` warnings are **expected** for SQL-only tables; column lineage through `raw_orders.amount` is still complete.

In [ ]:
from clearmetric.adapters import ingest_all
from clearmetric.compiler import build_graph, compile, discover
from clearmetric.core import load_project_config
from clearmetric.emitters import emit_compile

project = load_project_config(PROJECT_DIR)
print("=== discover ===")
for src in discover(PROJECT_DIR).sources:
    print(f"{src.kind}\t{src.path}")

print("\n=== ingest_all ===")
for kind, artifact in ingest_all(project):
    print(f"{kind}: nodes={len(artifact.nodes)} edges={len(artifact.edges)} warnings={len(artifact.warnings)}")

built = build_graph(PROJECT_DIR)
print("\n=== merged graph (text) ===")
print(emit_compile("text", built))

## Compile (enforce)

`compile()` runs posture checks and returns the enforced graph used by impact and emitters.

In [ ]:
compiled = compile(PROJECT_DIR)
artifact = compiled.artifact
print(f"nodes: {len(artifact.nodes)}  edges: {len(artifact.edges)}  warnings: {len(artifact.warnings)}")

## Impact preview

Selection strings use `table.column`. Ground truth for this fixture:

| Selection | Direction | Related columns |
|-----------|-----------|------------------|
| `customers_report.customer_lifetime_value` | upstream | 3 |
| `orders_base.amount` | downstream | 2 |

Notebook **03** goes deeper on impact semantics and renderers.

In [ ]:
from clearmetric.compiler.impact import impact

UPSTREAM = {"column:customer_totals.total_amount", "column:orders_base.amount", "column:raw_orders.amount"}
DOWNSTREAM = {"column:customer_totals.total_amount", "column:customers_report.customer_lifetime_value"}

_, upstream = impact(PROJECT_DIR, selection="customers_report.customer_lifetime_value", direction="upstream")
_, downstream = impact(PROJECT_DIR, selection="orders_base.amount", direction="downstream")

assert set(upstream.related_ids) == UPSTREAM
assert set(downstream.related_ids) == DOWNSTREAM
print("upstream:", sorted(upstream.related_ids))
print("downstream:", sorted(downstream.related_ids))